# Model Training — PMS Task Overdue Prediction

Two dataset variants explored side-by-side:
1. **Creation-time** (28 features after feature selection)
2. **Halfway** (40 features — creation + accumulation)

Split: **70% train / 15% val / 15% test**. Test is held out until final evaluation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from copy import deepcopy
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, precision_recall_curve, auc
)
from xgboost import XGBClassifier
from catboost import CatBoostClassifier, Pool
from lightgbm import LGBMClassifier
import optuna

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
%matplotlib inline

RANDOM_STATE = 42
N_TRIALS = 50
print('Libraries loaded.')

---
## 1. Load Datasets

In [ ]:
DATA_DIR = Path('../../data/v1')

df_creation = pd.read_csv(DATA_DIR / 'dataset_at_creation_fixed_end_date.csv')
df_halfway  = pd.read_csv(DATA_DIR / 'dataset_at_halfway_fixed_end_date.csv')

print(f'Creation-time: {df_creation.shape[0]} rows, {df_creation.shape[1]} cols')
print(f'Halfway:       {df_halfway.shape[0]} rows, {df_halfway.shape[1]} cols')

In [ ]:
assert df_creation['id'].equals(df_halfway['id']), 'Task IDs must match'
assert df_creation['calculated_overdue'].equals(df_halfway['calculated_overdue']), 'Targets must match'
print('IDs and targets are consistent.')

In [ ]:
target = 'calculated_overdue'
id_col = 'id'

X_creation = df_creation.drop(columns=[id_col, target])
X_halfway  = df_halfway.drop(columns=[id_col, target])
y = df_creation[target]

creation_feats = list(X_creation.columns)
halfway_feats  = list(X_halfway.columns)
added_feats    = [c for c in halfway_feats if c not in creation_feats]

print(f'Creation features: {len(creation_feats)}')
print(f'Halfway features:  {len(halfway_feats)}')
print(f'Added at halfway:  {len(added_feats)}')

## 2. Target Distribution

In [ ]:
target_dist = y.value_counts(normalize=True)
print(f'Overdue rate: {target_dist[1]:.1%}')
print(f'Not overdue:  {target_dist[0]:.1%}')

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
y.value_counts().plot(kind='bar', ax=ax[0], color=['steelblue', 'coral'])
ax[0].set_title('Class Counts')
ax[0].set_xticklabels(['Not Overdue', 'Overdue'], rotation=0)
y.value_counts(normalize=True).plot(kind='bar', ax=ax[1], color=['steelblue', 'coral'])
ax[1].set_title('Class Proportions')
ax[1].set_xticklabels(['Not Overdue', 'Overdue'], rotation=0)
plt.tight_layout()
plt.show()

## 3. Feature Selection & Preprocessing

In [ ]:
DROPPED_BOTH = ['ma_comment_count', 'wl_low', 'kpi_comment_count']
DROPPED_HALFWAY = DROPPED_BOTH + ['subtask_completion_pct', 'subtask_overdue_rate']

X_creation = X_creation.drop(columns=[c for c in DROPPED_BOTH if c in X_creation.columns])
X_halfway  = X_halfway.drop(columns=[c for c in DROPPED_HALFWAY if c in X_halfway.columns])

if 'planned_duration' in X_creation.columns:
    X_creation['planned_duration'] = X_creation['planned_duration'].clip(lower=0)
if 'planned_duration' in X_halfway.columns:
    X_halfway['planned_duration'] = X_halfway['planned_duration'].clip(lower=0)
if 'num_ma_revisions' in X_creation.columns:
    X_creation['num_ma_revisions'] = X_creation['num_ma_revisions'].clip(upper=78)
if 'num_ma_revisions' in X_halfway.columns:
    X_halfway['num_ma_revisions'] = X_halfway['num_ma_revisions'].clip(upper=78)
if 'num_revisions' in X_creation.columns:
    X_creation['num_revisions'] = X_creation['num_revisions'].clip(upper=9)
if 'num_revisions' in X_halfway.columns:
    X_halfway['num_revisions'] = X_halfway['num_revisions'].clip(upper=9)

challenge_flags = ['has_challenges', 'has_kpi_challenge', 'has_kpi_potential_challenge', 'has_subtask_challenge']
existing = [c for c in challenge_flags if c in X_halfway.columns]
if existing:
    X_creation['total_challenge_load'] = X_creation[[c for c in existing if c in X_creation.columns]].sum(axis=1)
    X_halfway['total_challenge_load'] = X_halfway[existing].sum(axis=1)

print(f'Creation: {X_creation.shape[1]} features')
print(f'Halfway:  {X_halfway.shape[1]} features')

## 4. Train / Val / Test Split (70 / 15 / 15)

We split once and reuse the same indices across both dataset variants.

In [ ]:
Xc_train, Xc_temp, Xh_train, Xh_temp, y_train, y_temp = train_test_split(
    X_creation, X_halfway, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)
Xc_val, Xc_test, Xh_val, Xh_test, y_val, y_test = train_test_split(
    Xc_temp, Xh_temp, y_temp, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp
)

print(f'Train:  {Xh_train.shape[0]} ({Xh_train.shape[0]/len(y):.0%})')
print(f'Val:    {Xh_val.shape[0]} ({Xh_val.shape[0]/len(y):.0%})')
print(f'Test:   {Xh_test.shape[0]} ({Xh_test.shape[0]/len(y):.0%})')
print(f'\nTrain target:\n{y_train.value_counts(normalize=True)}')

## 5. Baseline Models

Default params on **halfway** dataset. Evaluated on validation set (test untouched).

In [ ]:
def evaluate_model(model, X_train, X_val, y_train, y_val, name='Model'):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)[:, 1]
    return {
        'model': name,
        'accuracy': accuracy_score(y_val, y_pred),
        'precision': precision_score(y_val, y_pred, zero_division=0),
        'recall': recall_score(y_val, y_pred, zero_division=0),
        'f1': f1_score(y_val, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_val, y_proba),
    }

In [ ]:
default_models = {
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                              random_state=RANDOM_STATE, eval_metric='logloss'),
    'CatBoost': CatBoostClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                                    random_state=RANDOM_STATE, verbose=0),
    'LightGBM': LGBMClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                                random_state=RANDOM_STATE, verbose=0),
}

baseline_results = []
for name, model in default_models.items():
    r = evaluate_model(deepcopy(model), Xh_train, Xh_val, y_train, y_val, f'{name} (Halfway)')
    baseline_results.append(r)

pd.DataFrame(baseline_results)

---
## 6. Optuna Hyperparameter Tuning

5-fold CV on the **training set** (70% of data). After tuning, the best model is selected by validation set performance.

In [ ]:
def make_objective(model_name, X, y):
    def objective(trial):
        if model_name == 'XGBoost':
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=50),
                'max_depth': trial.suggest_int('max_depth', 3, 12),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'subsample': trial.suggest_float('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
                'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
                'gamma': trial.suggest_float('gamma', 0, 5),
                'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10, log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10, log=True),
                'random_state': RANDOM_STATE,
                'eval_metric': 'logloss',
            }
            model = XGBClassifier(**params)
        elif model_name == 'CatBoost':
            params = {
                'iterations': trial.suggest_int('iterations', 100, 500, step=50),
                'depth': trial.suggest_int('depth', 4, 10),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
                'border_count': trial.suggest_int('border_count', 32, 255),
                'random_seed': RANDOM_STATE,
                'verbose': 0,
            }
            model = CatBoostClassifier(**params)
        else:
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=50),
                'max_depth': trial.suggest_int('max_depth', 3, 12),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'num_leaves': trial.suggest_int('num_leaves', 15, 127),
                'subsample': trial.suggest_float('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
                'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
                'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10, log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10, log=True),
                'random_state': RANDOM_STATE,
                'verbose': 0,
            }
            model = LGBMClassifier(**params)

        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
        scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
        return scores.mean()
    return objective

In [ ]:
tuned_models = {}
optuna_records = []

for model_name in ['XGBoost', 'CatBoost', 'LightGBM']:
    print(f'\n=== Tuning {model_name} ===')
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.optimize(make_objective(model_name, Xh_train, y_train), n_trials=N_TRIALS, show_progress_bar=True)

    best_params = study.best_params
    print(f'Best CV ROC-AUC: {study.best_value:.4f}')

    fixed_params = {'random_state': RANDOM_STATE, 'verbose': 0}
    if model_name == 'XGBoost':
        fixed_params['eval_metric'] = 'logloss'
    if model_name == 'CatBoost':
        fixed_params['random_seed'] = RANDOM_STATE
        fixed_params.pop('random_state')

    model_class = {'XGBoost': XGBClassifier, 'CatBoost': CatBoostClassifier, 'LightGBM': LGBMClassifier}[model_name]
    model = model_class(**{**best_params, **fixed_params})
    tuned_models[model_name] = model
    optuna_records.append({'model': model_name, 'best_cv_roc_auc': study.best_value})

pd.DataFrame(optuna_records)

In [ ]:
# Evaluate each tuned model on the validation set to pick the best
val_results = []
for name, model in tuned_models.items():
    r = evaluate_model(deepcopy(model), Xh_train, Xh_val, y_train, y_val, name)
    val_results.append(r)

val_results_df = pd.DataFrame(val_results)
val_results_df

In [ ]:
best_model_name = val_results_df.sort_values('f1', ascending=False).iloc[0]['model']
print(f'Best model (by val F1): {best_model_name}')

## 7. Feature Importance (Best Model)

In [ ]:
def plot_feature_importance(model, feature_names, title, top_n=15):
    importances = pd.Series(model.feature_importances_, index=feature_names)
    importances = importances.sort_values(ascending=False).head(top_n)
    fig, ax = plt.subplots(figsize=(10, max(6, top_n * 0.35)))
    importances.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(title)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
    return importances

best_clf = deepcopy(tuned_models[best_model_name])
best_clf.fit(Xh_train, y_train)
imp = plot_feature_importance(best_clf, Xh_train.columns, f'Top Features — {best_model_name}', top_n=15)

---
## 8. Fold-Safe Training

The features `position_id_encoded`, `dept_past_overdue_rate`, `dept_avg_revisions`, `emp_past_overdue_rate`, `pos_past_overdue_rate` were computed using the full-dataset target, leaking validation information. We quantify the gap.

In [ ]:
LEAKY_FEATURES = ['position_id_encoded', 'dept_past_overdue_rate', 'dept_avg_revisions',
                  'emp_past_overdue_rate', 'pos_past_overdue_rate']
LEAKY_FEATURES = [c for c in LEAKY_FEATURES if c in Xh_train.columns]

Xh_train_fs = Xh_train.drop(columns=LEAKY_FEATURES)
Xh_val_fs   = Xh_val.drop(columns=LEAKY_FEATURES)
Xh_test_fs  = Xh_test.drop(columns=LEAKY_FEATURES)

print(f'Full features: {Xh_train.shape[1]}  |  Fold-safe: {Xh_train_fs.shape[1]}')
print(f'Removed: {LEAKY_FEATURES}')

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

fold_results = []
for name, clf_class, kwargs in [
    ('XGBoost', XGBClassifier, {'random_state': RANDOM_STATE, 'eval_metric': 'logloss'}),
    ('CatBoost', CatBoostClassifier, {'random_seed': RANDOM_STATE, 'verbose': 0}),
    ('LightGBM', LGBMClassifier, {'random_state': RANDOM_STATE, 'verbose': 0}),
]:
    leaky, safe = [], []
    for train_idx, val_idx in cv.split(Xh_train, y_train):
        m = clf_class(**kwargs)
        m.fit(Xh_train.iloc[train_idx], y_train.iloc[train_idx])
        leaky.append(roc_auc_score(y_train.iloc[val_idx], m.predict_proba(Xh_train.iloc[val_idx])[:, 1]))

        m = clf_class(**kwargs)
        m.fit(Xh_train_fs.iloc[train_idx], y_train.iloc[train_idx])
        safe.append(roc_auc_score(y_train.iloc[val_idx], m.predict_proba(Xh_train_fs.iloc[val_idx])[:, 1]))

    leaky, safe = np.array(leaky), np.array(safe)
    fold_results.append({
        'model': name, 'leaky_mean': leaky.mean(), 'leaky_std': leaky.std(),
        'safe_mean': safe.mean(), 'safe_std': safe.std(), 'gap': leaky.mean() - safe.mean(),
    })
    print(f'{name}: Leaky={leaky.mean():.4f}  Safe={safe.mean():.4f}  Gap={leaky.mean()-safe.mean():.4f}')

fold_results_df = pd.DataFrame(fold_results)
fold_results_df

---
## 9. Final Model — Train on Train+Val, Evaluate on Test

Retrain the best tuned model on **train + val** using fold-safe features. Evaluate on the never-before-seen **test** set. Track loss curves during training.

In [ ]:
X_train_final = pd.concat([Xh_train_fs, Xh_val_fs], axis=0)
y_train_final = pd.concat([y_train, y_val], axis=0)

print(f'Final training set: {X_train_final.shape}')
print(f'Test set:           {Xh_test_fs.shape}')

In [ ]:
# Retrain the best model with loss tracking
best_params = tuned_models[best_model_name].get_params()

if best_model_name == 'XGBoost':
    best_params['eval_metric'] = 'logloss'
    best_params['early_stopping_rounds'] = 20
    final_model = XGBClassifier(**best_params)
    final_model.fit(
        X_train_final, y_train_final,
        eval_set=[(Xh_val_fs, y_val)],
        verbose=0
    )
    results = final_model.evals_result()
    train_loss = results['validation_0']['logloss']

elif best_model_name == 'CatBoost':
    best_params.pop('verbose', None)
    final_model = CatBoostClassifier(**best_params, verbose=0, eval_metric='Logloss')
    final_model.fit(
        X_train_final, y_train_final,
        eval_set=(Xh_val_fs, y_val),
    )
    evals = final_model.get_evals_result()
    train_loss = evals['learn']['Logloss']
    val_loss = evals['validation']['Logloss']

else:  # LightGBM
    best_params['early_stopping_rounds'] = 20
    final_model = LGBMClassifier(**best_params)
    final_model.fit(
        X_train_final, y_train_final,
        eval_set=[(Xh_val_fs, y_val)],
    )
    train_loss = final_model.evals_result_['training'][final_model.objective_ if hasattr(final_model, 'objective_') else 'binary_logloss']

y_pred = final_model.predict(Xh_test_fs)
y_proba = final_model.predict_proba(Xh_test_fs)[:, 1]

### 9.1 Loss Curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

if hasattr(final_model, 'evals_result') or best_model_name == 'CatBoost':
    evals = final_model.get_evals_result() if best_model_name == 'CatBoost' else final_model.evals_result()
    if 'validation' in evals:
        metric = list(evals['validation'].keys())[0]
        ax.plot(evals['validation'][metric], label=f'Val {metric}', color='coral')
    if 'learn' in evals:
        metric = list(evals['learn'].keys())[0]
        ax.plot(evals['learn'][metric], label=f'Train {metric}', color='steelblue')
elif hasattr(final_model, 'evals_result_'):
    for ds_name, metrics in final_model.evals_result_.items():
        for metric, values in metrics.items():
            ax.plot(values, label=f'{ds_name} {metric}')

ax.set_xlabel('Boosting Round')
ax.set_ylabel('Loss')
ax.set_title(f'Training Loss — {best_model_name}')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

### 9.2 Test Set Evaluation

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Not Overdue', 'Overdue'],
            yticklabels=['Not Overdue', 'Overdue'])
plt.title(f'{best_model_name} — Test Set (Fold-Safe)')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

print(classification_report(y_test, y_pred, target_names=['Not Overdue', 'Overdue']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}')

prec, rec, _ = precision_recall_curve(y_test, y_proba)
print(f'PR-AUC: {auc(rec, prec):.4f}')

---
## Summary

| Step | Detail |
|------|--------|
| Split | 70% train / 15% val / 15% test (stratified) |
| Baseline | Default XGBoost/CatBoost/LightGBM → val set |
| Optuna | 50 trials × 3 models, 5-fold CV on training set |
| Best model | Picked by F1 on validation set |
| Fold-safe | Dropped 5 leakage-prone features, quantified gap |
| Final | Retrained best model on train+val, tested on held-out test |

**Next:** Error analysis, time-based holdout, model deployment.